# 3. Modelo Base (M1, Entrega 1)

Este notebook ajusta e compara os candidatos do Modelo Base:
- Regressão logística com WoE (scorecard)
- LightGBM
- XGBoost

Detalhes:
- Aqui são usadas apenas as colunas fornecidas na base original
- Usamos novamente o método de usar 3 sementes nos boostings
- A OOT interna só é usada para confirmar o modelo escolhido no fim

## 3.1 Imports, leitura e conjuntos

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from equifax.ajuste import ajustar_boosting, ajustar_logistica_woe, comparar_balanceamento, dobras_por_id
from equifax.avaliacao import (grafico_calibracao, grafico_distribuicao_score, grafico_ks_por_safra,
                               grafico_roc, psi_score_por_safra, simular_politica, tabela_calibracao,
                               tabela_decis)
from equifax.config import RANDOM_SEED, REPORTS, TABLES
from equifax.conjuntos import carregar_conjunto, listar_conjuntos
from equifax.dados import carregar_desenvolvimento, janelas
from equifax.experimentos import (escorar, importancia, ler_registro, resumir, rodar_experimento,
                                  treinar)
from equifax.modelos import tabela_pontos
from equifax.validacao import avaliar
from equifax.viz import salvar_figura

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

In [ ]:
# Conjunto final do funil (02) e tempo máximo de cada busca
CONJUNTO = "proposta_242"
# Número de tentativas fixo (sem limite de tempo), para a busca sair igual em qualquer máquina.
# São as tentativas que couberam em 40 minutos na rodada de 18/10: com a mesma semente, o Optuna
# repete as mesmas tentativas e chega aos mesmos parâmetros.
N_TENTATIVAS = {"lgbm": 23, "xgb": 8}

VARIAVEIS = carregar_conjunto(CONJUNTO)
print(len(VARIAVEIS), "variáveis;", "estado" in VARIAVEIS)
listar_conjuntos()

In [ ]:
base = carregar_desenvolvimento()
partes = janelas(base)  # treino e teste; a OOT interna não é carregada aqui
treino, teste = partes["treino"], partes["teste"]
for nome, parte in partes.items():
    print(f"{nome:7} n={len(parte):7} inadimplência={1 - parte['adimplencia'].mean():.4f}")

# Mesmas 4 dobras para todos os modelos
DOBRAS = dobras_por_id(treino, n_dobras=4)

## 3.2 Referência

Usamos o LightGBM sem ajuste, com os parâmetros do `02_selecao_variaveis`, como a linha de base

In [ ]:
rodar_experimento("m1_lgbm_referencia", CONJUNTO, partes)
resumir(ler_registro(), nomes=["m1_lgbm_referencia"])

## 3.3 Regressão logística com WoE

Cada variável vira o WoE da faixa em que cai (binning ótimo, o mesmo da ficha, ajustado no treino de cada dobra). Saem as de IV abaixo de 0,02 e uma de cada grupo de WoE correlacionados. A grade testa a força da regularização (`C`), L2 (`l1_ratio = 0`) ou L1 (`l1_ratio = 1`) e o limiar de correlação (0,7 ou 0,8, mais baixo que o 0,90 da seleção, como é comum em scorecards).

In [ ]:
params_logit, ajuste_logit = ajustar_logistica_woe(treino, VARIAVEIS, dobras=DOBRAS)
ajuste_logit.to_csv(TABLES / "ajuste_m1_logit_woe.csv", index=False)
print("Melhores parâmetros:", params_logit)
ajuste_logit.head(10).round(4)

In [ ]:
# A logística é determinística: uma semente basta
rodar_experimento("m1_logit_woe", CONJUNTO, partes, modelo="logit_woe", params=params_logit,
                  sementes=(RANDOM_SEED,))

logit = treinar("logit_woe", params_logit, treino, VARIAVEIS)
transformador = logit.estimador.transformador_
print("Convergiu:", logit.estimador.convergiu_)
print(len(VARIAVEIS), "variáveis de entrada ->", len(transformador.variaveis_), "depois dos filtros ->",
      int((logit.estimador.coeficientes != 0).sum()), "com coeficiente diferente de zero")
pd.Series(transformador.removidas_).str.split(" ").str[0].value_counts()

In [ ]:
# Scorecard: pontos por faixa (600 pontos = chance de 50 bons para 1 mau; a chance dobra a cada 20 pontos)
pontos = tabela_pontos(logit.estimador)
pontos.to_csv(TABLES / "scorecard_m1_logit_woe.csv", index=False)
efeito = importancia(logit).sort_values(ascending=False)
print("As 15 variáveis de maior efeito (|coeficiente| x desvio do WoE):")
display(efeito.head(15).round(4))
pontos[pontos["variavel"].isin(efeito.index[:3])]

## 3.4 LightGBM

Busca com Optuna \
Cada tentativa é avaliada pela AUC média nas 4 dobras do treino, com parada antecipada na dobra. Nulos e códigos especiais ficam como estão. O peso de classe (`balancear`) entra na busca e depois é comparado à parte com os melhores parâmetros.

In [ ]:
params_lgbm, ajuste_lgbm = ajustar_boosting("lgbm", treino, VARIAVEIS, n_tentativas=N_TENTATIVAS["lgbm"],
                                            timeout=None, dobras=DOBRAS)
ajuste_lgbm.to_csv(TABLES / "ajuste_m1_lgbm.csv", index=False)
print("Melhores parâmetros:", params_lgbm)
ajuste_lgbm.head(10).round(4)

In [ ]:
comparar_balanceamento("lgbm", params_lgbm, treino, VARIAVEIS, dobras=DOBRAS).round(4)

## 3.5 XGBoost

In [ ]:
params_xgb, ajuste_xgb = ajustar_boosting("xgb", treino, VARIAVEIS, n_tentativas=N_TENTATIVAS["xgb"],
                                          timeout=None, dobras=DOBRAS)
ajuste_xgb.to_csv(TABLES / "ajuste_m1_xgb.csv", index=False)
print("Melhores parâmetros:", params_xgb)
ajuste_xgb.head(10).round(4)

In [ ]:
comparar_balanceamento("xgb", params_xgb, treino, VARIAVEIS, dobras=DOBRAS).round(4)

## 3.6 Candidatos no teste

Os quatro candidatos são treinados no treino inteiro e avaliados no teste, com 3 sementes nos boostings. A escolha considera AUC, KS, o pior KS entre as safras e a simplicidade

In [ ]:
rodar_experimento("m1_lgbm_ajustado", CONJUNTO, partes, modelo="lgbm", params=params_lgbm)
rodar_experimento("m1_xgb_ajustado", CONJUNTO, partes, modelo="xgb", params=params_xgb)

CANDIDATOS = ["m1_lgbm_referencia", "m1_logit_woe", "m1_lgbm_ajustado", "m1_xgb_ajustado"]
registro = ler_registro()
registro = registro[registro["conjunto"] == CONJUNTO]
resumir(registro, nomes=CANDIDATOS)

In [ ]:
# Parâmetros dos candidatos, versionados (só hiperparâmetros, nenhum dado)
with open(REPORTS / "params_m1.json", "w", encoding="utf-8") as f:
    json.dump({"conjunto": CONJUNTO, "logit_woe": params_logit, "lgbm": params_lgbm, "xgb": params_xgb},
              f, ensure_ascii=False, indent=2)

## 3.7 Avaliação detalhada no teste

Para a logística e para os dois boostings ajustados (semente 42): KS por safra, distribuição do score por safra, ROC, decis, PSI do score, calibração e simulação de política

In [ ]:
modelos_finais = {"Logística WoE": logit,
                  "LightGBM": treinar("lgbm", params_lgbm, treino, VARIAVEIS),
                  "XGBoost": treinar("xgb", params_xgb, treino, VARIAVEIS)}
scores = {nome: escorar(m, teste) for nome, m in modelos_finais.items()}
ARQUIVO = {"Logística WoE": "logistica", "LightGBM": "lightgbm", "XGBoost": "xgboost"}  # nomes sem acento
y_teste, safra_teste = teste["adimplencia"], teste["safra"]

metricas = {nome: avaliar(y_teste, s, safra_teste) for nome, s in scores.items()}
grafico_ks_por_safra(metricas, "KS por safra no teste (M1)")
salvar_figura("m1_ks_por_safra")
plt.show()

grafico_roc({nome: (y_teste, s) for nome, s in scores.items()}, "Curva ROC no teste (M1)")
salvar_figura("m1_roc")
plt.show()

In [ ]:
for nome, s in scores.items():
    grafico_distribuicao_score(s, safra_teste, f"Distribuição do score por safra no teste: {nome}")
    salvar_figura(f"m1_distribuicao_score_{ARQUIVO[nome]}")
    plt.show()

In [ ]:
pd.DataFrame({nome: psi_score_por_safra(s, safra_teste) for nome, s in scores.items()}).round(4)

In [ ]:
for nome, s in scores.items():
    print(nome)
    display(tabela_decis(y_teste, s).round(4))

In [ ]:
for nome, s in scores.items():
    tabela, brier = tabela_calibracao(y_teste, s)
    print(f"{nome}: Brier {brier:.4f}")
    grafico_calibracao(y_teste, s, f"Calibração no teste: {nome}")
    salvar_figura(f"m1_calibracao_{ARQUIVO[nome]}")
    plt.show()

In [ ]:
# Política: aprovar os clientes de maior score até cada taxa de aprovação
pd.concat({nome: simular_politica(y_teste, s) for nome, s in scores.items()}, axis=1).round(4)

> Conclusão omitida na versão pública: ela traz números e achados da base, que é confidencial.